# Rodar tudo no Google Colab — Case Meridian
Este notebook: **(1)** recebe o pacote do projeto, **(2)** baixa os dados, **(3)** executa os dois notebooks de análise e **(4)** abre o app Streamlit com um link público temporário.

**Como usar:** `Ambiente de execução → Executar tudo`. Quando pedir, envie o arquivo `repo_meridian.zip`.
> O link do app (célula 5) funciona só enquanto a sessão do Colab estiver aberta.

In [ ]:
# 1) Enviar e descompactar o pacote do projeto
import os, zipfile
from google.colab import files
up = files.upload()                       # escolha repo_meridian.zip
nome = next(n for n in up if n.endswith('.zip'))
zipfile.ZipFile(nome).extractall('/content')
REPO = '/content/repo'
assert os.path.isdir(REPO), 'Pasta repo/ não encontrada dentro do zip'
print('Projeto em', REPO, os.listdir(REPO))

In [ ]:
# 2) Dependências e dados
!pip -q install streamlit kagglehub statsmodels
import kagglehub, os
DATA = kagglehub.dataset_download('nadhirmazariboufares/supply-chain-dataset')
os.environ['CASE_DATA_DIR'] = DATA
print('Dados em', DATA, '|', len([f for f in os.listdir(DATA) if f.endswith('.csv')]), 'CSVs')

In [ ]:
# 3) Executar os dois notebooks (salva as versões executadas em /content/saida)
!mkdir -p /content/saida
!cd {REPO}/notebook && jupyter nbconvert --to notebook --execute Case_Meridian_Supply_Chain.ipynb --output-dir /content/saida --output principal_executado.ipynb --ExecutePreprocessor.timeout=900
!cd {REPO}/notebook && jupyter nbconvert --to notebook --execute Case_Meridian_Atrasos_Fornecedores.ipynb --output-dir /content/saida --output atrasos_executado.ipynb --ExecutePreprocessor.timeout=1500

In [ ]:
# 4) Conferir se algum notebook terminou com erro
import nbformat
for arq in ['principal_executado', 'atrasos_executado']:
    nb = nbformat.read(f'/content/saida/{arq}.ipynb', 4)
    erros = [i for i, c in enumerate(nb.cells) if c.cell_type == 'code' for o in c.outputs if o.output_type == 'error']
    print(arq, '→', 'OK, 0 erros' if not erros else f'ERROS nas células {erros}')

In [ ]:
# 5) Abrir o app Streamlit — opção A: proxy do próprio Colab
import subprocess, time, os
env = dict(os.environ, CASE_DATA_DIR=DATA)
subprocess.Popen(f'cd {REPO}/streamlit_app && streamlit run app.py --server.port 8501 --server.headless true --server.enableCORS false --server.enableXsrfProtection false > /content/streamlit.log 2>&1', shell=True, env=env)
time.sleep(15)
print(open('/content/streamlit.log').read()[-400:])
from google.colab import output
output.serve_kernel_port_as_window(8501)

**Se a opção A abrir em branco ou der erro de conexão**, rode a célula abaixo (opção B, localtunnel). Ela pressupõe que o app da célula 5 já está rodando.

In [ ]:
# 6) Opção B: localtunnel (a página pede uma senha: é o IP impresso abaixo)
!npm install -g localtunnel >/dev/null 2>&1
import urllib.request, subprocess, time
ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode().strip()
print('Senha da página do localtunnel (é o IP):', ip)
subprocess.Popen('lt --port 8501 > /content/lt.log 2>&1', shell=True); time.sleep(8)
print(open('/content/lt.log').read())